# Risk Stratified Table

**This notebook generates a Table 1 for patients with advanced urothelial cancer receiving first-line pembrolizumab versus carboplatin-based chemotherapy, showing covariate differences stratified by the threshold effect.** 

In [1]:
import numpy as np
import pandas as pd

from tableone import TableOne

## Import data

In [2]:
treatment_df = pd.read_csv('../outputs/pembro_carbo_index.csv')

In [3]:
treatment_df.sample(3)

,PatientID,LineName,StartDate,avelumab_maintenance
2700,FF97758679E0F,carbo,2018-01-10,0
3683,FCEBEA2C351EA,carbo,2011-02-24,0
2461,F930E69A140E7,carbo,2022-11-02,0


In [4]:
treatment_df.shape

(3712, 4)

In [5]:
treatment_df['treatment'] = (treatment_df['LineName'] == 'pembro').astype(int)

In [6]:
dtype_map = pd.read_csv('../outputs/pembro_carbo_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
features_df = pd.read_csv('../outputs/pembro_carbo_features_df.csv', dtype = dtype_map)

In [7]:
features_df.shape

(3706, 162)

In [8]:
surv_pred_df = pd.read_csv('../outputs/gb_6m_survival_predictions_calibrated.csv')

In [9]:
surv_pred_df.head(3)

,PatientID,psurv_180_calibrated
0,F5AAF96C85477,0.593639
1,F788831A66E9A,0.422425
2,FAF26C1A1CEE4,0.447253


In [10]:
surv_pred_df.shape

(3706, 2)

In [11]:
df = pd.merge(features_df, treatment_df, on = 'PatientID', how = 'left')

In [12]:
df.shape

(3706, 166)

In [13]:
df = pd.merge(df, surv_pred_df, on = 'PatientID', how = 'left')

In [14]:
df.shape

(3706, 167)

## Binning relevant covariates

In [15]:
df['weight_loss_5p'] = np.where(df['percent_change_weight'] <= -5, 1, 0)

In [16]:
df['creatinine_2'] = np.where(df['creatinine'] > 2, 1, 0)

In [17]:
df['hemoglobin_9'] = np.where(df['hemoglobin'] < 9, 1, 0)

In [18]:
df['ast_60'] = np.where(df['ast'] > 60, 1, 0)

In [19]:
df['albumin_3'] = np.where(df['albumin'] < 30, 1, 0)

In [20]:
df['alp_200'] = np.where(df['alp'] > 200, 1, 0)

In [21]:
df['above_threshold'] = np.where(df['psurv_180_calibrated'] > 0.84, 1, 0)

## Creating table 

In [22]:
table1 = TableOne(
    data = df, 
    columns = ['age', 'GroupStage_mod', 'SmokingStatus', 'Surgery_mod', 'PDL1_status', 'ecog_index', 'weight_loss_5p', 'creatinine_2', 'ast_60', 'hemoglobin_9', 'albumin_3', 'alp_200', 'liver_met', 'thoracic_met', 'bone_met'],
    categorical = ['GroupStage_mod', 'SmokingStatus', 'Surgery_mod', 'PDL1_status', 'ecog_index', 'weight_loss_5p', 'creatinine_2', 'ast_60', 'hemoglobin_9', 'albumin_3', 'alp_200', 'liver_met', 'thoracic_met', 'bone_met'],
    continuous = ['age'],
    nonnormal = ['age'],
    groupby = 'above_threshold',
    pval = True)

In [23]:
table1

Grouped by above_threshold                                                              
                                                                Missing           Overall                 0                 1 P-Value
n                                                                                    3706              2631              1075        
age, median [Q1,Q3]                                                   0  75.0 [68.0,81.0]  77.0 [69.0,82.5]  73.0 [67.0,78.0]  <0.001
GroupStage_mod, n (%) 0.0                                                        22 (0.6)          14 (0.5)           8 (0.7)  <0.001
                      1.0                                                        67 (1.8)          42 (1.6)          25 (2.3)        
                      2.0                                                       294 (7.9)         199 (7.6)          95 (8.8)        
                      3.0                                                       267 (7.2)         157 (6.0)        110 (10.2)        
                      4.0                                                     3056 (82.5)       2219 (84.3)        837 (77.9)        
SmokingStatus, n (%)  History of smoking                                      2736 (73.8)       1975 (75.1)        761 (70.8)   0.024
                      No history of smoking                                    945 (25.5)        638 (24.2)        307 (28.6)        
                      Unknown/not documented                                     25 (0.7)          18 (0.7)           7 (0.7)        
Surgery_mod, n (%)    0                                                       2048 (55.3)       1599 (60.8)        449 (41.8)  <0.001
                      1                                                       1658 (44.7)       1032 (39.2)        626 (58.2)        
PDL1_status, n (%)    negative                                                  294 (7.9)         219 (8.3)          75 (7.0)   0.338
                      positive                                                  144 (3.9)          99 (3.8)          45 (4.2)        
                      unknown                                                 3268 (88.2)       2313 (87.9)        955 (88.8)        
ecog_index, n (%)     0.0                                                      782 (21.1)        380 (14.4)        402 (37.4)  <0.001
                      1.0                                                     2315 (62.5)       1653 (62.8)        662 (61.6)        
                      2.0                                                      483 (13.0)        472 (17.9)          11 (1.0)        
                      3.0                                                       120 (3.2)         120 (4.6)           0 (0.0)        
                      4.0                                                         6 (0.2)           6 (0.2)           0 (0.0)        
weight_loss_5p, n (%) 0                                                       3003 (81.0)       2010 (76.4)        993 (92.4)  <0.001
                      1                                                        703 (19.0)        621 (23.6)          82 (7.6)        
creatinine_2, n (%)   0                                                       3304 (89.2)       2270 (86.3)       1034 (96.2)  <0.001
                      1                                                        402 (10.8)        361 (13.7)          41 (3.8)        
ast_60, n (%)         0                                                       3597 (97.1)       2532 (96.2)       1065 (99.1)  <0.001
                      1                                                         109 (2.9)          99 (3.8)          10 (0.9)        
hemoglobin_9, n (%)   0                                                       3343 (90.2)       2279 (86.6)       1064 (99.0)  <0.001
                      1                                                         363 (9.8)        352 (13.4)          11 (1.0)        
albumin_3, n (%)      0  

In [24]:
table1.to_csv('../outputs/risk_stratified_table1.csv', header = True)

In [25]:
df.groupby('above_threshold')['ecog_index_na'].sum()

above_threshold
0    814
1    316
Name: ecog_index_na, dtype: int64